## U-Net Building Blocks


In [38]:
""" Parts of the U-Net model """

import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        # if you have padding issues, see
        # https://github.com/HaiyongJiang/U-Net-Pytorch-Unstructured-Buggy/commit/0e854509c2cea854e247a9c615f175f76fbb2e3a
        # https://github.com/xiaopeng-liao/Pytorch-UNet/commit/8ebac70e633bac59fc22bb5195e513d5832fb3bd
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


## U-Net Model Assembly

In [39]:
""" Full assembly of the parts to form the complete network """

# from .unet_parts import *

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = (DoubleConv(n_channels, 64))
        self.down1 = (Down(64, 128))
        self.down2 = (Down(128, 256))
        self.down3 = (Down(256, 512))
        factor = 2 if bilinear else 1
        self.down4 = (Down(512, 1024 // factor))
        self.up1 = (Up(1024, 512 // factor, bilinear))
        self.up2 = (Up(512, 256 // factor, bilinear))
        self.up3 = (Up(256, 128 // factor, bilinear))
        self.up4 = (Up(128, 64, bilinear))
        self.outc = (OutConv(64, n_classes))

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

    def use_checkpointing(self):
        self.inc = torch.utils.checkpoint(self.inc)
        self.down1 = torch.utils.checkpoint(self.down1)
        self.down2 = torch.utils.checkpoint(self.down2)
        self.down3 = torch.utils.checkpoint(self.down3)
        self.down4 = torch.utils.checkpoint(self.down4)
        self.up1 = torch.utils.checkpoint(self.up1)
        self.up2 = torch.utils.checkpoint(self.up2)
        self.up3 = torch.utils.checkpoint(self.up3)
        self.up4 = torch.utils.checkpoint(self.up4)
        self.outc = torch.utils.checkpoint(self.outc)

## Model Loading

In [40]:
import torch

# Load a model state dictionary (weights only)
model = UNet(n_channels=1,n_classes=1) # Instantiate your model class
model.load_state_dict(torch.load('/kaggle/input/kaggle_0.0750/pytorch/default/1/Kaggle_0.0750.pth'))
model.eval() # Set the model to evaluation mode

# Load an entire saved model (including architecture and optimizer state)
# model = torch.load('path/to/entire_model.pth')


UNet(
  (inc): DoubleConv(
    (double_conv): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (down1): Down(
    (maxpool_conv): Sequential(
      (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1): DoubleConv(
        (double_conv): Sequential(
          (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
 

## Imports & Setup

In [41]:
# ----------------------
# Imports & Setup
# ----------------------
import os
import random
import glob
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")


## Custom Dataset

In [42]:
# ----------------------
# Gunshot Dataset
# ----------------------
class GunshotDataset(Dataset):
    def __init__(self, clean_dir, noise_dir, n_fft=512, hop_length=128,
                 duration=2.0, sr=16000):
        self.clean_files = glob.glob(os.path.join(clean_dir, "**", "*.wav"), recursive=True)
        self.noise_files = glob.glob(os.path.join(noise_dir, "**", "*.wav"), recursive=True)

        if len(self.clean_files) == 0:
            raise FileNotFoundError(f"No clean files found in {clean_dir}")
        if len(self.noise_files) == 0:
            raise FileNotFoundError(f"No noise files found in {noise_dir}")

        self.n_fft = n_fft
        self.hop_length = hop_length
        self.duration = duration
        self.sr = sr

        print(f"Loaded {len(self.clean_files)} clean and {len(self.noise_files)} noise files")

    def __len__(self):
        return len(self.clean_files)

    def __getitem__(self, idx):
        clean, sr = torchaudio.load(self.clean_files[idx])
        noise, sr_n = torchaudio.load(random.choice(self.noise_files))

        clean, noise = clean.mean(dim=0, keepdim=True), noise.mean(dim=0, keepdim=True)

        if sr != self.sr:
            clean = torchaudio.transforms.Resample(sr, self.sr)(clean)
        if sr_n != self.sr:
            noise = torchaudio.transforms.Resample(sr_n, self.sr)(noise)

        num_samples = int(self.duration * self.sr)
        if clean.size(1) < num_samples:
            clean = torch.nn.functional.pad(clean, (0, num_samples - clean.size(1)))
        else:
            clean = clean[:, :num_samples]

        if noise.size(1) < num_samples:
            noise = torch.nn.functional.pad(noise, (0, num_samples - noise.size(1)))
        else:
            start = random.randint(0, noise.size(1) - num_samples)
            noise = noise[:, start:start + num_samples]

        snr_db = random.uniform(0, 10)
        factor = clean.norm(p=2) / (10 ** (snr_db / 20) * noise.norm(p=2) + 1e-8)
        noisy = clean + factor * noise

        spec = torchaudio.transforms.Spectrogram(n_fft=self.n_fft, hop_length=self.hop_length, power=None)
        return spec(noisy).abs(), spec(clean).abs()


## Training Function

In [43]:
# ----------------------
# Training Loop
# ----------------------
def train(train_loader, val_loader, model, device,
          patience=15, max_epochs=200,
          save_dir="/kaggle/working/checkpoints"):

    os.makedirs(save_dir, exist_ok=True)

    criterion = nn.L1Loss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

    start_epoch = 1
    best_val = float("inf")
    checkpoint_path = os.path.join(save_dir, "checkpoint.pth")

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        start_epoch = checkpoint['epoch'] + 1
        best_val = checkpoint.get('best_val', float("inf"))
        print(f"🔄 Resumed from epoch {checkpoint['epoch']} (best val={best_val:.4f})")

    no_improve = 0
    train_losses, val_losses = [], []

    for epoch in range(start_epoch, max_epochs + 1):
        model.train()
        total_train = 0
        for noisy, clean in train_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            optimizer.zero_grad()
            out = model(noisy)
            loss = criterion(out, clean)
            loss.backward()
            optimizer.step()
            total_train += loss.item()
        train_loss = total_train / len(train_loader)

        model.eval()
        total_val = 0
        with torch.no_grad():
            for noisy, clean in val_loader:
                noisy, clean = noisy.to(device), clean.to(device)
                out = model(noisy)
                total_val += criterion(out, clean).item()
        val_loss = total_val / len(val_loader)

        scheduler.step(val_loss)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Epoch {epoch}: Train {train_loss:.4f} | Val {val_loss:.4f}")

        torch.save(model.state_dict(), os.path.join(save_dir, f"unet_epoch{epoch}.pth"))
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_val': best_val
        }, checkpoint_path)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), os.path.join(save_dir, "best_model.pth"))
            no_improve = 0
            print("✅ New best model saved")
        else:
            no_improve += 1
            print(f"⏳ No improvement for {no_improve} epochs")

        if no_improve >= patience:
            print("⛔ Early stopping triggered")
            break

    print("🏁 Training complete!")

    return train_losses, val_losses


## Plot Function

In [44]:
# ----------------------
# Plot Training History
# ----------------------
def plot_history(train_losses, val_losses):
    plt.figure(figsize=(8,5))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


## Dataset & Training Pipeline

In [ ]:
# ----------------------
# Imports
# ----------------------
import os
import random
import glob
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import Dataset, DataLoader
# from unet.unet_model import UNet
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ----------------------
# Dataset
# ----------------------
class GunshotDataset(Dataset):
    def __init__(self, clean_dir, noise_dir, n_fft=512, hop_length=128,
                 duration=2.0, sr=16000):
        # Recursively find all wav files
        self.clean_files = glob.glob(os.path.join(clean_dir, "**", "*.wav"), recursive=True)
        self.noise_files = glob.glob(os.path.join(noise_dir, "**", "*.wav"), recursive=True)

        # Check if files are found
        if len(self.clean_files) == 0:
            raise FileNotFoundError(f"No clean files found in {clean_dir}")
        if len(self.noise_files) == 0:
            raise FileNotFoundError(f"No noise files found in {noise_dir}")

        self.n_fft = n_fft
        self.hop_length = hop_length
        self.duration = duration
        self.sr = sr

        print(f"Loaded {len(self.clean_files)} clean and {len(self.noise_files)} noise files")

    def __len__(self):
        return len(self.clean_files)

    def __getitem__(self, idx):
        clean, sr = torchaudio.load(self.clean_files[idx])
        noise, sr_n = torchaudio.load(random.choice(self.noise_files))

        clean, noise = clean.mean(dim=0, keepdim=True), noise.mean(dim=0, keepdim=True)

        if sr != self.sr:
            clean = torchaudio.transforms.Resample(sr, self.sr)(clean)
        if sr_n != self.sr:
            noise = torchaudio.transforms.Resample(sr_n, self.sr)(noise)

        num_samples = int(self.duration * self.sr)
        if clean.size(1) < num_samples:
            clean = torch.nn.functional.pad(clean, (0, num_samples - clean.size(1)))
        else:
            clean = clean[:, :num_samples]

        if noise.size(1) < num_samples:
            noise = torch.nn.functional.pad(noise, (0, num_samples - noise.size(1)))
        else:
            start = random.randint(0, noise.size(1) - num_samples)
            noise = noise[:, start:start + num_samples]

        snr_db = random.uniform(0, 10)
        factor = clean.norm(p=2) / (10 ** (snr_db / 20) * noise.norm(p=2) + 1e-8)
        noisy = clean + factor * noise

        spec = torchaudio.transforms.Spectrogram(n_fft=self.n_fft, hop_length=self.hop_length, power=None)
        return spec(noisy).abs(), spec(clean).abs()


# ----------------------
# Training
# ----------------------
def train(train_loader, val_loader, model, device,
          patience=15, max_epochs=200,
          save_dir="/kaggle/working/checkpoints"):

    os.makedirs(save_dir, exist_ok=True)

    criterion = nn.L1Loss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    # 🔑 Scheduler added
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

    start_epoch = 1
    best_val = float("inf")
    checkpoint_path = os.path.join(save_dir, "checkpoint.pth")

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        start_epoch = checkpoint['epoch'] + 1
        best_val = checkpoint.get('best_val', float("inf"))
        print(f"🔄 Resumed from epoch {checkpoint['epoch']} (best val={best_val:.4f})")

    no_improve = 0
    train_losses, val_losses = [], []

    for epoch in range(start_epoch, max_epochs + 1):
        model.train()
        total_train = 0
        for noisy, clean in train_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            optimizer.zero_grad()
            out = model(noisy)
            loss = criterion(out, clean)
            loss.backward()
            optimizer.step()
            total_train += loss.item()
        train_loss = total_train / len(train_loader)

        model.eval()
        total_val = 0
        with torch.no_grad():
            for noisy, clean in val_loader:
                noisy, clean = noisy.to(device), clean.to(device)
                out = model(noisy)
                total_val += criterion(out, clean).item()
        val_loss = total_val / len(val_loader)

        # 🔑 Step scheduler
        scheduler.step(val_loss)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Epoch {epoch}: Train {train_loss:.4f} | Val {val_loss:.4f}")

        torch.save(model.state_dict(), os.path.join(save_dir, f"unet_epoch{epoch}.pth"))
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_val': best_val
        }, checkpoint_path)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), os.path.join(save_dir, "best_model.pth"))
            no_improve = 0
            print("✅ New best model saved")
        else:
            no_improve += 1
            print(f"⏳ No improvement for {no_improve} epochs")

        if no_improve >= patience:
            print("⛔ Early stopping triggered")
            break

    print("🏁 Training complete!")

    plt.figure(figsize=(8,5))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


# ----------------------
# Main
# ----------------------
if __name__ == "__main__":
    base = "/kaggle/input/gunshot-dataset/gunshot_dataset"  # ✅ fixed path

    train_ds = GunshotDataset(os.path.join(base, "train/clean"),
                              os.path.join(base, "train/noise"))
    val_ds = GunshotDataset(os.path.join(base, "val/clean"),
                            os.path.join(base, "val/noise"))

    # 🔑 Increased batch size + drop_last for val
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, drop_last=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = UNet(n_channels=1, n_classes=1).to(device)

    train(train_loader, val_loader, model, device)


Loaded 2296 clean and 3971 noise files
Loaded 736 clean and 993 noise files
🔄 Resumed from epoch 83 (best val=0.0619)


In [34]:
!ls /kaggle/working/checkpoints


best_model.pth	  unet_epoch29.pth  unet_epoch49.pth  unet_epoch69.pth
checkpoint.pth	  unet_epoch2.pth   unet_epoch4.pth   unet_epoch6.pth
unet_epoch10.pth  unet_epoch30.pth  unet_epoch50.pth  unet_epoch70.pth
unet_epoch11.pth  unet_epoch31.pth  unet_epoch51.pth  unet_epoch71.pth
unet_epoch12.pth  unet_epoch32.pth  unet_epoch52.pth  unet_epoch72.pth
unet_epoch13.pth  unet_epoch33.pth  unet_epoch53.pth  unet_epoch73.pth
unet_epoch14.pth  unet_epoch34.pth  unet_epoch54.pth  unet_epoch74.pth
unet_epoch15.pth  unet_epoch35.pth  unet_epoch55.pth  unet_epoch75.pth
unet_epoch16.pth  unet_epoch36.pth  unet_epoch56.pth  unet_epoch76.pth
unet_epoch17.pth  unet_epoch37.pth  unet_epoch57.pth  unet_epoch77.pth
unet_epoch18.pth  unet_epoch38.pth  unet_epoch58.pth  unet_epoch78.pth
unet_epoch19.pth  unet_epoch39.pth  unet_epoch59.pth  unet_epoch79.pth
unet_epoch1.pth   unet_epoch3.pth   unet_epoch5.pth   unet_epoch7.pth
unet_epoch20.pth  unet_epoch40.pth  unet_epoch60.pth  unet_epoch80.pth
unet_epoch

In [35]:
import shutil

shutil.copy("/kaggle/working/checkpoints/best_model.pth", "/kaggle/working/best_model.pth")


'/kaggle/working/best_model.pth'